In [37]:
import pandas as pd
import numpy as np

수집된 용어 csv 전처리

In [38]:
file_name = "./input/term_data.csv"

df = pd.read_csv(file_name)

print(df.head())
print(df.tail())

          English Word                                    English Meaning  \
0          A/B Testing  A method of comparing two versions of a webpag...   
1       Above the Fold  The content visible on a screen without scroll...   
2  Acceptance Criteria  Predefined requirements that must be met for a...   
3        Accessibility  The design of products and services to be usab...   
4           Actionable  Data or insight that can be used to inform a d...   

   Korean Word                                     Korean Meaning  
0      A/B 테스트  두 가지 이상의 시안 중 어느 것이 더 나은 성과를 보이는지 사용자의 반응을 통해 ...  
1  스크롤 없는 첫 화면                 사용자가 스크롤을 내리지 않고 처음으로 보게 되는 화면 영역.  
2        인수 기준               개발된 기능이 요구사항을 충족했는지 판단하는 구체적인 조건 목록.  
3          접근성    장애 여부와 관계없이 누구나 동등하게 정보에 접근하고 이용할 수 있도록 보장하는 것.  
4       실행 가능한                실제 행동이나 의사결정으로 이어질 수 있는, 구체적이고 유용한.  
           English Word                                    English Meaning  \
598            Zero-day  A vulnerability that is ex

In [39]:
# 인덱스 번호 넣기

index_numbers = np.arange(1, len(df) + 1)
df.insert(loc=0, column='def_id', value=index_numbers)
print(df.head())

   def_id         English Word  \
0       1          A/B Testing   
1       2       Above the Fold   
2       3  Acceptance Criteria   
3       4        Accessibility   
4       5           Actionable   

                                     English Meaning  Korean Word  \
0  A method of comparing two versions of a webpag...      A/B 테스트   
1  The content visible on a screen without scroll...  스크롤 없는 첫 화면   
2  Predefined requirements that must be met for a...        인수 기준   
3  The design of products and services to be usab...          접근성   
4  Data or insight that can be used to inform a d...       실행 가능한   

                                      Korean Meaning  
0  두 가지 이상의 시안 중 어느 것이 더 나은 성과를 보이는지 사용자의 반응을 통해 ...  
1                 사용자가 스크롤을 내리지 않고 처음으로 보게 되는 화면 영역.  
2               개발된 기능이 요구사항을 충족했는지 판단하는 구체적인 조건 목록.  
3    장애 여부와 관계없이 누구나 동등하게 정보에 접근하고 이용할 수 있도록 보장하는 것.  
4                실제 행동이나 의사결정으로 이어질 수 있는, 구체적이고 유용한.  


In [40]:
df_definition = df[['def_id', 'English Meaning', 'Korean Meaning']].copy()

print(df_definition.head())

   def_id                                    English Meaning  \
0       1  A method of comparing two versions of a webpag...   
1       2  The content visible on a screen without scroll...   
2       3  Predefined requirements that must be met for a...   
3       4  The design of products and services to be usab...   
4       5  Data or insight that can be used to inform a d...   

                                      Korean Meaning  
0  두 가지 이상의 시안 중 어느 것이 더 나은 성과를 보이는지 사용자의 반응을 통해 ...  
1                 사용자가 스크롤을 내리지 않고 처음으로 보게 되는 화면 영역.  
2               개발된 기능이 요구사항을 충족했는지 판단하는 구체적인 조건 목록.  
3    장애 여부와 관계없이 누구나 동등하게 정보에 접근하고 이용할 수 있도록 보장하는 것.  
4                실제 행동이나 의사결정으로 이어질 수 있는, 구체적이고 유용한.  


In [41]:
df_term = df[['def_id', 'English Word', 'Korean Word']].copy()

print(df_term.head())

   def_id         English Word  Korean Word
0       1          A/B Testing      A/B 테스트
1       2       Above the Fold  스크롤 없는 첫 화면
2       3  Acceptance Criteria        인수 기준
3       4        Accessibility          접근성
4       5           Actionable       실행 가능한


In [42]:
df_term_renamed = df_term.rename(columns={
    'English Word': 'English',
    'Korean Word': 'Korean'
}).copy()

df_melted = df_term_renamed.melt(
    id_vars=['def_id'],                            # 유지할 기준 열
    value_vars=['English', 'Korean'],    # 행으로 변환할 대상 열
    var_name='Language',                           # 임시 열 (Eng/Kor 구분용)
    value_name='Word'                              # 모든 단어가 담길 새 열 이름
)
df_melted = df_melted.dropna(subset=['Word'])

df_melted

,def_id,Language,Word
0,1,English,A/B Testing
1,2,English,Above the Fold
2,3,English,Acceptance Criteria
3,4,English,Accessibility
4,5,English,Actionable
...,...,...,...
1201,599,Korean,제로데이 공격
1202,600,Korean,무중단
1203,601,Korean,좀비 프로세스
1204,602,Korean,존 (Zone)


In [43]:
df_exploded = df_melted.assign(
    Word=df_melted['Word'].str.split(r'[()]')
).explode('Word')

# 3a. 분리 과정에서 생긴 불필요한 공백을 .str.strip()으로 제거합니다.
df_final_term = df_exploded.assign(Word=df_exploded['Word'].str.strip())

# 3b. 공백 제거 후 남은 빈 문자열('')을 np.nan (결측치)으로 변경합니다.
df_final_term['Word'].replace('', np.nan, inplace=True)

# 3c. 빈 문자열(이제 NaN)이 된 행을 .dropna()로 완전히 제거합니다.
df_final_term = df_final_term.dropna(subset=['Word'])

df_final_term = df_final_term.drop_duplicates().sort_values('def_id').reset_index(drop=True)

# --- 4. 결과 확인 ---
print("\n--- 최종 결과 (df_final_term) ---")
print(df_final_term.head(10))

df_final_term


--- 최종 결과 (df_final_term) ---
   def_id Language                 Word
0       1  English          A/B Testing
1       1   Korean              A/B 테스트
2       2   Korean          스크롤 없는 첫 화면
3       2  English       Above the Fold
4       3  English  Acceptance Criteria
5       3   Korean                인수 기준
6       4   Korean                  접근성
7       4  English        Accessibility
8       5   Korean               실행 가능한
9       5  English           Actionable


C:\Users\USER\AppData\Local\Temp\ipykernel_6368\1622778819.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_final_term['Word'].replace('', np.nan, inplace=True)


,def_id,Language,Word
0,1,English,A/B Testing
1,1,Korean,A/B 테스트
2,2,Korean,스크롤 없는 첫 화면
3,2,English,Above the Fold
4,3,English,Acceptance Criteria
...,...,...,...
1302,602,Korean,Zone
1303,602,Korean,존
1304,602,English,Zone
1305,603,English,Zoom in / Zoom out


In [44]:
import os

save_directory = r"C:\git_repo\workspace\talaria-ai\data_pipeline\output"
os.makedirs(save_directory, exist_ok=True)

file_path_terms = os.path.join(save_directory, "terms.csv")
file_path_definitions = os.path.join(save_directory, "definitions.csv")

df_final_term.to_csv(file_path_terms, index=False, encoding='utf-8-sig')
df_definition.to_csv(file_path_definitions, index=False, encoding='utf-8-sig')

print(f"CSV 파일 2개가 다음 경로에 저장되었습니다:\n{save_directory}")

CSV 파일 2개가 다음 경로에 저장되었습니다:
C:\git_repo\workspace\talaria-ai\data_pipeline\output
